# Exact counting at any cardinality

"How many distinct values does this column have?" is the most expensive
question in profiling when answered naively, and onceover answers it
exactly at any cardinality. The trick is not a sketch — it is choosing
between two exact algorithms by watching the workload, and keeping the
value keys compact enough that both algorithms are cheap.

## The value key

Counting operates on `VKey`, a compact enumerable form: 64-bit integers
as-is; floats as their bit patterns (NaN excluded, −0.0 normalized —
two f64s are equal iff their bits are equal); strings; booleans. `VKey`
is ordered — by value within a variant, so a million distinct integers
sort as integers, not as strings — and the order is total, which is what
makes the sorted-run fallback deterministic under parallelism.

The card keeps only what it needs: the exact distinct count over all
non-null values, the non-null row count, and the top 64 values by
(count descending, key ascending). High-cardinality sorting borrows
string views rather than owning them — only the reported top values
ever own their strings — so a 10-million-distinct string column does not
become a 10-million-allocation column.

## Two algorithms, one exact answer

`build_counts` starts with a hash map (`rustc_hash`'s FxHashMap — a
non-cryptographic hasher chosen for speed; nothing in counting depends
on collision resistance because nothing in counting trusts the hash
for equality). It accumulates exact counts until the map holds 4,096
distinct keys. If the column has more, the routine switches: sort the
compact keys, walk the sorted array once, and count runs. Both paths
examine every row; both produce the exact distinct count; the limit
chooses the algorithm, never the accuracy.

The rule of thumb the threshold encodes: a hash map wins while
*distinct values* are few, because it is O(n) with a small constant;
sorting wins when distinct values are numerous, because it is
O(n log n) with no per-insert allocation and perfect cache behavior on
compact keys. A column of a million distinct i64s sorts in a few
hundred megabytes of borrowed slices; a hash map of a million entries
would rather more, and would have grown there one rehash at a time.

## Concentration in the same pass

HHI, entropy, and effective cardinality are accumulated *while*
counting — over the full value distribution, not a top-k sample — so
`effective_cardinality` on a million-distinct column is as exact as the
distinct count beside it. This is also where determinism lives: the
hash-map path sorts its entries by key before accumulating the
concentration summary, so the floating-point summation order is fixed
regardless of hash-table layout or thread count ([](determinism.ipynb)).

The same machinery feeds everything downstream: key discovery reads the
exact distinct counts; role inference reads them with the null counts;
the FD screen hashes the same values further along; the duplicates pass
compares canonical bytes. One exact counting core, many consumers — and
every number on every card is the same number a verification pass would
recompute.